
# COPY INTO aand MERGE Commands

### Copy Into command
##### - Incrementally loads data into delta lake tables from cloud storage
##### - supports schema evolution
##### - supports wide range of file formats(CSV, JSOM, Parquet, Delta)
##### - Alternative to Auto Loader for batch ingeston

### Documentation https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into

##### Create the table to copy the data into

In [0]:
create table gizmobox_catalog_noori.delta_lake.raw_stock_prices_2

##### Incrementally load new files into the table


In [0]:
Delete from gizmobox_catalog_noori.delta_lake.raw_stock_prices_2;

copy into gizmobox_catalog_noori.delta_lake.raw_stock_prices_2
from 'abfss://databrickspracticecontainer@databricksstgnoori.dfs.core.windows.net/Landing/stock_prices'
fileformat = JSON
format_options('inferSchema' = 'true')
copy_options ('mergeSchema' = 'true')

In [0]:
select * from gizmobox_catalog_noori.delta_lake.raw_stock_prices_2

## Merge Statement

##### - used for upserts (Insert / update/ Delete operations in a single statement)
##### - Allows merging new data into a target table based on matching condition

Documentation - https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-merge-into


### Create the table to merge the data into

In [0]:
create table if not exists gizmobox_catalog_noori.delta_lake.stock_prices_2
(   
    stock_id string,    
    price double,
    trading_date string
)

### Merge the source data into target table
##### - 1. Insert new stocks recieved
##### - 2. Update price and trading_date if updates receieved
##### - 3. Delete stocks which are de-listed from the exchange (status ='Delisted')

In [0]:
merge into gizmobox_catalog_noori.delta_lake.stock_prices_2 as target
using gizmobox_catalog_noori.delta_lake.raw_stock_prices_2 as source
on target.stock_id = source.stock_id
when matched and source.status='ACTIVE'  then 
    update set target.price = source.price, target.trading_date = source.trading_date
when matched and source.status='DELISTED' then
    delete
when not matched and source.status='ACTIVE' then 
    insert (stock_id, price, trading_date) values (source.stock_id, source.price, source.trading_date)



In [0]:
select * from gizmobox_catalog_noori.delta_lake.stock_prices_2

In [0]:
desc history gizmobox_catalog_noori.delta_lake.stock_prices_2